In [8]:
import numpy
import pandas as pd
import os
from src.data_loader import prepare_paths, load_horizontal_well, load_typewell
from src.features import (generate_signal_features, generate_spatial_features,
        integrate_with_typewells, full_preprocessing_pipeline)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from src.prepare_data import prepare_target
from src.prepare_data import train_test_split_paths
from sklearn.metrics import root_mean_squared_error
from src.model_utils import validate_model

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [9]:
PATH_TRAIN = 'train'
PATH_TEST = 'test'

paths_train = prepare_paths(PATH_TRAIN)
paths_test = prepare_paths(PATH_TEST)

train_X, train_y, val_X, val_y = train_test_split_paths(paths_train)

test = []

for path in paths_test: #only 3 pairs
    new_df = full_preprocessing_pipeline(*path)
    test.append(new_df)

processed 100 .csv's
processed 200 .csv's
processed 300 .csv's
processed 400 .csv's
processed 500 .csv's
processed 600 .csv's
processed 700 .csv's
processed 773 .csv's


In [ ]:
model_xgb = xgb.XGBRegressor(
    n_estimators=900, #
    max_depth=5,
    learning_rate=0.02, 
    subsample=0.9,
    colsample_bytree=0.9,
    colsample_bylevel=0.9,
    min_child_weight=3,
    reg_alpha=1.0,
    reg_lambda=2.0,
    gamma=0.1,
    max_bin=256,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

In [11]:
concated_train_X = pd.concat(train_X, ignore_index=True)
concated_train_y = pd.concat(train_y, ignore_index=True)
concated_val_X = pd.concat(val_X, ignore_index=True)
concated_val_y = pd.concat(val_y, ignore_index=True)

In [12]:
model_xgb.fit(concated_train_X, concated_train_y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=0.9, colsample_bynode=None, colsample_bytree=0.9,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric='rmse', feature_types=None, gamma=0.1,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.02, max_bin=256,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=3, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=900, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [13]:
validate_model(model_xgb, concated_train_X, concated_train_y,
                concated_val_X, concated_val_y)

train rmse: 6.297176141811139
val rmse: 8.424099410397686


(6.297176141811139, 8.424099410397686)

In [14]:
model_xgb.save_model('xgb_model.json')

## Best model xgb

- params: n_estimators=900, max_depth=5, learning_rate=0.02, subsample=0.9,
    colsample_bytree=0.9, colsample_bylevel=0.9, min_child_weight=3, reg_alpha=1.0,
    reg_lambda=2.0, gamma=0.1, max_bin=256, objective="reg:squarederror", eval_metric="rmse",
    tree_method="hist", random_state=42, n_jobs=-1,

- train rmse score: 6.297176141811139
- val rmse score: 8.424099410397686

In [19]:
test[0].head()

,MD,X,Y,Z,GR,TVT_input,gr_roll_mean_5,gr_roll_std_5,gr_roll_min_5,gr_roll_max_5,...,gr_lead_2,gr_lag_3,gr_lead_3,delta_X,delta_Y,delta_Z,trajectory_slope,tw_reference_GR,tw_reference_TVT,gr_difference_to_tw
0,11467.0,2983525.16,1069022.09,-9258.57,115.692586,11236.02,122.241280,11.436583,115.584293,135.446960,...,135.446960,115.692586,140.401346,0.00,0.00,0.00,0.000000,115.53,11340.45,0.162586
1,11468.0,2983525.18,1069022.30,-9259.55,115.584293,11237.05,126.781296,13.024744,115.584293,140.401346,...,140.401346,115.692586,111.270638,0.02,0.21,-0.98,-4.645624,115.53,11340.45,0.054293
2,11469.0,2983525.20,1069022.52,-9260.52,135.446960,11238.09,123.679165,13.241943,111.270638,140.401346,...,111.270638,115.692586,108.779909,0.02,0.22,-0.97,-4.390964,135.14,11275.45,0.306960
3,11470.0,2983525.22,1069022.73,-9261.50,140.401346,11239.12,122.296629,14.577737,108.779909,140.401346,...,108.779909,115.692586,114.338929,0.02,0.21,-0.98,-4.645624,139.44,11252.95,0.961346
4,11471.0,2983525.25,1069022.95,-9262.47,111.270638,11240.15,122.047556,14.730928,108.779909,140.401346,...,114.338929,115.584293,121.459167,0.03,0.22,-0.97,-4.368641,111.25,11318.45,0.020638


In [ ]:
from sklearn.linear_model import Ridge

correction_model = Ridge()
